# Extensión del semillerío del 9105 a k=50

Este notebook agrega **30 semillas nuevas** al semillerío del 9105 (reusa las 20 que ya están guardadas en `./semillas/`), y sube el ensemble k=50 a Kaggle en varios cortes.

**Objetivo:** reducir la varianza esperada en Private del ~0.9 (k=20) al ~0.55 (k=50). No sube el techo pero baja el piso.

**Requisito:** el pipeline del 9105 debe haber corrido hasta `dfinal_train`, `param_final`, `campos_buenos`, `mfuture` y `dfuture`. Si esa sesión sigue viva, arrancá desde la celda 1. Si la perdiste, correr primero el 9105 hasta la celda que arma `param_final`.

In [ ]:
# --- Sanity check: los objetos del pipeline 9105 están en memoria? ---
require("data.table")
require("lightgbm")

obligatorios <- c("dfinal_train", "param_final", "campos_buenos", "mfuture", "dfuture")
faltantes <- obligatorios[!sapply(obligatorios, exists)]

if (length(faltantes) > 0) {
  stop("Faltan objetos en memoria: ", paste(faltantes, collapse = ", "),
       "\nCorré el notebook 9105 hasta la celda que arma param_final antes de continuar.")
}

cat("Todos los objetos necesarios están en memoria. OK para continuar.\n")
cat("Filas de dfinal_train:", dfinal_train$dim()[1], "\n")
cat("Filas de mfuture:     ", nrow(mfuture), "\n")
cat("num_iterations final: ", param_final$num_iterations, "\n")
cat("num_leaves final:     ", param_final$num_leaves, "\n")
cat("learning_rate final:  ", param_final$learning_rate, "\n")
cat("(esperados: niter=347, leaves=256, lr=0.01318)\n")

In [ ]:
# --- Definición de las 50 semillas: 20 originales del 9105 + 30 nuevas ---

PARAM$semillerio$semillas_k20 <- c(
  # las 5 originales
  804043, 653561, 703903, 439693, 665857,
  # las 15 primeras nuevas (ya corridas en el 9105)
  246319, 719179, 688511, 678859, 759179,
  748567, 319687, 771091, 684007, 514853,
  377749, 329977, 757927, 724837, 216973
)

# 30 semillas nuevas del banco de primos
PARAM$semillerio$semillas_nuevas30 <- c(
  287333, 656119, 694919, 189817, 867463,
  791801, 804317, 680831, 330917, 595951,
  777571, 662339, 202667, 526159, 509389,
  865993, 749471, 398833, 153269, 969637,
  374683, 678481, 799333, 687083, 941207,
  213349, 735659, 872843, 803207, 414913
)

PARAM$semillerio$semillas_k50 <- c(
  PARAM$semillerio$semillas_k20,
  PARAM$semillerio$semillas_nuevas30
)

stopifnot(length(PARAM$semillerio$semillas_k50) == 50)
stopifnot(length(unique(PARAM$semillerio$semillas_k50)) == 50)

cat("Total de semillas en el ensemble k=50:",
    length(PARAM$semillerio$semillas_k50), "\n")

# chequeo cuántas semillas ya están en disco
dir.create("semillas", showWarnings = FALSE)
ya_entrenadas <- sapply(PARAM$semillerio$semillas_k50, function(s) {
  file.exists(paste0("semillas/prediccion_semilla_", s, ".txt"))
})

cat("Ya en disco: ", sum(ya_entrenadas), "\n")
cat("A entrenar:  ", sum(!ya_entrenadas), "\n")
cat("(esperado: 20 en disco, 30 a entrenar)\n")

In [ ]:
# --- Loop de entrenamiento de las 30 semillas que faltan ---
# Idempotente: si el archivo ya existe, salta.
# Tiempo estimado: ~50-70 min (30 semillas x 1.5-2.5 min c/u)

semillas_pendientes <- PARAM$semillerio$semillas_k50[!ya_entrenadas]

for (i in seq_along(semillas_pendientes)) {

  semilla <- semillas_pendientes[i]
  cat(format(Sys.time(), "%X"),
      " - Nueva semilla ", i, "/", length(semillas_pendientes),
      " = ", semilla, "\n", sep = "")

  param_semilla <- param_final
  param_semilla$seed <- semilla

  modelo_i <- lgb.train(
    data = dfinal_train,
    param = param_semilla,
    verbose = -100
  )

  prob_i <- predict(modelo_i, mfuture)

  tb_pred_i <- dfuture[, list(numero_de_cliente)]
  tb_pred_i[, prob := prob_i]
  fwrite(tb_pred_i,
    file = paste0("semillas/prediccion_semilla_", semilla, ".txt"),
    sep = "\t"
  )

  rm(modelo_i, prob_i, tb_pred_i)
  gc(full = TRUE, verbose = FALSE)
}

cat("\nEntrenamiento completado. Las 50 semillas están en disco.\n")

In [ ]:
# --- Cargar las 50 predicciones y armar el ensemble k=50 ---

tb_probs50 <- dfuture[, list(numero_de_cliente)]

for (semilla in PARAM$semillerio$semillas_k50) {
  tb_ind <- fread(paste0("semillas/prediccion_semilla_", semilla, ".txt"))
  tb_ind <- tb_ind[match(tb_probs50$numero_de_cliente, tb_ind$numero_de_cliente)]
  col_semilla <- paste0("prob_", semilla)
  tb_probs50[, (col_semilla) := tb_ind$prob]
}

cols_prob50 <- grep("^prob_", colnames(tb_probs50), value = TRUE)
stopifnot(length(cols_prob50) == 50)

tb_prediccion_k50 <- tb_probs50[, list(numero_de_cliente)]
tb_prediccion_k50[, prob := rowMeans(tb_probs50[, ..cols_prob50])]

fwrite(tb_prediccion_k50, file = "prediccion_k50.txt", sep = "\t")
fwrite(tb_probs50, file = "probs_por_semilla_k50.txt", sep = "\t")

cat("Ensemble k=50 armado y guardado.\n")

In [ ]:
# --- Análisis local: comparación k=20 vs k=50 ---

PARAM$kaggle$corte_referencia <- 2000  # el corte con máximo Public del 9105 k=20 (88.64)

# top-2000 del ensemble k=50
tb_pred50_sorted <- copy(tb_prediccion_k50)
setorder(tb_pred50_sorted, -prob)
top_k50 <- tb_pred50_sorted[1:PARAM$kaggle$corte_referencia, numero_de_cliente]

# top-2000 del ensemble k=20 (solo las 20 originales)
cols_prob20 <- paste0("prob_", PARAM$semillerio$semillas_k20)
tb_pred20 <- tb_probs50[, list(numero_de_cliente)]
tb_pred20[, prob := rowMeans(tb_probs50[, ..cols_prob20])]
setorder(tb_pred20, -prob)
top_k20 <- tb_pred20[1:PARAM$kaggle$corte_referencia, numero_de_cliente]

en_ambos <- length(intersect(top_k20, top_k50))
cat("Clientes en el top-2000 tanto en k=20 como en k=50:", en_ambos, "/",
    PARAM$kaggle$corte_referencia,
    "(", round(en_ambos / PARAM$kaggle$corte_referencia, 3), ")\n")

# correlación media entre las 50 semillas
mat_probs50 <- as.matrix(tb_probs50[, ..cols_prob50])
cor_matrix50 <- cor(mat_probs50)
cor_media50 <- mean(cor_matrix50[upper.tri(cor_matrix50)])
cat("Correlación media entre 50 semillas:", round(cor_media50, 4), "\n")

# distribución de votos
for (col in cols_prob50) {
  tb_probs50[, (paste0("in_top_", col)) :=
               numero_de_cliente %in% {
                 tb_tmp <- tb_probs50[, list(numero_de_cliente, p = get(col))]
                 setorder(tb_tmp, -p)
                 tb_tmp[1:PARAM$kaggle$corte_referencia, numero_de_cliente]
               }]
}
cols_in_top <- grep("^in_top_", colnames(tb_probs50), value = TRUE)
tb_probs50[, votos := rowSums(.SD), .SDcols = cols_in_top]

cat("\nDistribución de votos (top clientes por # de semillas que los eligieron):\n")
print(tb_probs50[votos > 0, .N, by = votos][order(-votos)])

In [ ]:
# --- Submit del ensemble k=50 a Kaggle ---

PARAM$kaggle$competencia <- "data-mining-junior-2026-a"
PARAM$kaggle$cortes <- seq(1800, 2400, by = 100)

setorder(tb_prediccion_k50, -prob)
dir.create("kaggle", showWarnings = FALSE)

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion_k50[, Predicted := 0L]
  tb_prediccion_k50[1:envios, Predicted := 1L]

  archivo_kaggle <- paste0("./kaggle/KA9105_FEratios_k50_", envios, ".csv")

  fwrite(tb_prediccion_k50[, list(numero_de_cliente, Predicted)],
         file = archivo_kaggle, sep = ",")

  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste("-f", archivo_kaggle)
  mensaje <- paste0("-m '9105 FE+ratios k=50 envios=", envios, "'")

  linea <- paste(comando, competencia, arch, mensaje)

  cat(format(Sys.time(), "%X"), " - submit k=50 envios=", envios, "\n", sep = "")
  salida <- system(linea, intern = TRUE)
  Sys.sleep(30)
  cat(salida, "\n")
}

cat("\nSubmits del ensemble k=50 completados. Comparalos con los del k=20 del 9105.\n")